# SAT-13B · Re-score every saved prediction on the corrected test labels  (CPU, ~20-40 min, no retraining)

**Attach:** `sat-ext` + `sat-splits` + `satelite-coco` + **`sat-test-relabeled`** + every `sat-results-*` Dataset
(+ `sat-splits-extra`). **Accelerator:** None. Run all / Save Version.

Check the log says `matched test images: 82 of 82`. Output -> New Dataset **`sat-results-13`** (tables T17, T18).
Then re-run **04, 09, 12** with `sat-results-13` attached too (logs print "runs use the CORRECTED test labels").

**Input setup** - right panel -> *Add Input* -> search the name -> *Add*. Links (your own Datasets are private: open them from the account that owns them, or share them with this account first):

* `sat-ext` - code (upload the sat_ext folder of the zip): https://www.kaggle.com/datasets/rifat4018/sat-ext
* `satelite-coco` - Roboflow COCO export, 640x640 (or download: universe.roboflow.com/rifat4018/aerial-object-detection-u3f5u): https://www.kaggle.com/datasets/rifat4018/satelite-coco
* `sat-splits` - output of notebook 01: https://www.kaggle.com/datasets/rifat4018/sat-splits
* `sat-splits-extra` - output of notebook 06 (optional): https://www.kaggle.com/datasets/rifat4018/sat-splits-extra
* `sat-results-13` - outputs of earlier notebooks (Output tab of that notebook -> *New Dataset*, same name); search them under *Your Datasets*.
* Check the printed input tree in the next cell before running the rest.

In [ ]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

In [ ]:
utils.warn_if_gpu_wasted()
import relabel, report, dataset as ds
sdir = ds.find_split_dir("grouped")
gt_old = utils.load_json(os.path.join(sdir, "coco", "test.json"))
info = ds.materialize("grouped")
new_boxes = relabel.load_relabeled(gt_old, info["images"]["test"])
gt_new = relabel.corrected_gt(gt_old, new_boxes)
chg = relabel.change_report(gt_old, gt_new)
os.makedirs(report.PAPER_DIR, exist_ok=True)
report.save(chg, "T17_test_relabel_changes", "Test-set annotation correction (notebook 13): boxes per class and size before and "
            "after adding the missing objects (same 82 images).")
relabel.write_corrected_splits(gt_new)
t = relabel.rescore_all(gt_new)
t.to_csv(os.path.join(report.PAPER_DIR, "T18_rescore_old_vs_new.csv"), index=False)
s = t[t["mode"].isin(["standard", "sahi480", "sahi320", "sahi160", "tta", "res1024"])]
print(s.groupby(["model", "mode"])[["AP_old", "AP_new", "P_old", "P_new", "R_old", "R_new", "FP_old", "FP_new"]].mean().round(3).to_string())
import shutil
shutil.rmtree(info["root"], ignore_errors=True)

In [ ]:
import shutil
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, "results_13"), "zip", cfg.WORK_DIR, "results")
print(sorted(os.listdir(cfg.WORK_DIR)), z)